In [86]:
import requests
import pandas as pd
import json
from dotenv import load_dotenv
import os
load_dotenv()
import sys

# NASA Data Research

In [96]:
# Setting up a batch size. I know this is not the best way to do it, but it works without any issues
credentials = os.getenv('NASA_ACCESS_KEY')
url = 'https://api.nasa.gov/neo/rest/v1/neo/browse?api_key=' + credentials
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    print(f"Initial request failed with status code: {response.status_code}")
    sys.exit(0)

# Now we are exporting all NEOs to a csv file
for _ in range(data['page']['total_pages']):
    # We have to be able to stop our script at any point if it crashes or something and continue from where we left off
    try:
        if str(_) in pd.read_csv('../data/nasa_data/all_neos.csv', usecols=['page'])['page'].values:
            print('skipped',_)
            continue
    except:
        pass

    # Creating a link
    url = f'https://api.nasa.gov/neo/rest/v1/neo/browse?page={_}&size=20&api_key=' + credentials
    print('loading page ' + str(_))
    
    # It's just more understandable this way cmon
    response = requests.get(url)
    data = response.json()

    filtered_data = pd.DataFrame(data['near_earth_objects']).drop('links', axis=1) # Dropping the links column as a security concern
    filtered_data['page'] = _ #Adding a page column to the dataframe

    # Exporting the data
    if response.status_code == 200:
        if _ == 0: # If it's the first page, we write the header and creating a file basically
            filtered_data.to_csv('../data/nasa_data/all_neos.csv', mode='w', index=False, header=True)
        else:
            filtered_data.to_csv('../data/nasa_data/all_neos.csv', mode='a', index=False, header=False)
    else:
        print(f"Request failed with status code: {response.status_code}")
        break

skipped 0
skipped 1
skipped 2
skipped 3
skipped 4
skipped 5
skipped 6
skipped 7
skipped 8
skipped 9
skipped 10
loading page 11
loading page 12
skipped 13
skipped 14
loading page 15
loading page 16
loading page 17
loading page 18
loading page 19
loading page 20
loading page 21
loading page 22
loading page 23
loading page 24
loading page 25
loading page 26
loading page 27
loading page 28
loading page 29
loading page 30
loading page 31
loading page 32
loading page 33
loading page 34
loading page 35
loading page 36
loading page 37
loading page 38
loading page 39
loading page 40
loading page 41
loading page 42
loading page 43
loading page 44
loading page 45
loading page 46
loading page 47
loading page 48
loading page 49
loading page 50
loading page 51
loading page 52
loading page 53
loading page 54
loading page 55
loading page 56
loading page 57
loading page 58
loading page 59
loading page 60
loading page 61
loading page 62
loading page 63
loading page 64
loading page 65
loading page 66
loa

In [ ]:
# Creating a DataFrame for all NEOs
filtered_data = pd.DataFrame(data['near_earth_objects'])
filtered_data.drop('links', axis=1, inplace=True) # Dropping links for security reasons



# Converting estimated diameter to meters and getting it from json
all_neos_df = filtered_data.copy()
all_neos_df['estimated_diameter_meters_max'] = all_neos_df.estimated_diameter.apply(lambda x: x['kilometers']['estimated_diameter_max'] * 1000)
all_neos_df['estimated_diameter_meters_min'] = all_neos_df.estimated_diameter.apply(lambda x: x['kilometers']['estimated_diameter_min'] * 1000)
all_neos_df.drop('estimated_diameter', axis = 1, inplace=True) # dropping json as we got all we needed

# Creating orbital data as a separate DataFrame. We will not parse it since we don't know if we will need it
orbital_data = all_neos_df[['id', 'neo_reference_id', 'name', 'name_limited', 'orbital_data']].copy()

# Creating close approach data as a separate DataFrame
close_approach_data = pd.DataFrame()

for i in range(len(all_neos_df)):
    temp_df = pd.DataFrame(all_neos_df.close_approach_data[i])
    temp_df['neo_reference_id'] = all_neos_df.neo_reference_id[i]
    temp_df['name_limited'] = all_neos_df.name_limited[i]
    temp_df['id'] = all_neos_df.id[i]
    temp_df['name'] = all_neos_df.name[i]
    temp_df['is_potentially_hazardous_asteroid'] = all_neos_df.is_potentially_hazardous_asteroid[i]
    try:
        temp_df['relative_velocity_kph'] = temp_df.relative_velocity.apply(lambda x: x['kilometers_per_hour'])
        temp_df['miss_distance_meters'] = temp_df.miss_distance.apply(lambda x: x['kilometers'] * 1000)
        close_approach_data = pd.concat([close_approach_data, temp_df])
    except:
        pass

# At the end we will get full data as all_neos_df